---
# Curso: Data Mining - Semana 6
Docente: Soledad Espezúa (s.espezual@up.edu.pe)

---

<a name='manejoarchivos'></a>
# <font color=blue>Laboratorio: tratamiento de valores faltantes y codificación de variables</font>

**Caso:** pacientes, atenciones, medicamentos y detalle de medicación.

#### Objetivos:
1. reconstruir una base integrada con unidad de análisis **paciente**;
2. distinguir entre tratar faltantes **antes** y **después** de una agregación;
3. evaluar eliminación (`dropna`) e imputación sin aplicarlas mecánicamente;
4. reconocer un faltante que no conviene imputar automáticamente;
5. codificar variables binarias y nominales;
6. construir una **primera matriz analítica**.

> En esta sesión **no** trabajaremos outliers, creación de nuevas variables derivadas, discretización ni escalamiento.

## <font color=green>1. Cargar las cuatro tablas</font>

Primero realizamos una limpieza estructural mínima: eliminar filas o columnas completamente vacías y duplicados exactos.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Configuramos un estilo simple para los gráficos.
sns.set_theme(style="whitegrid")

# Cargamos las cuatro tablas del caso clínico.
pacientes = pd.read_excel("Pacientes.xlsx")
atenciones = pd.read_excel("Atenciones.xlsx")
medicamentos = pd.read_excel("Medicamentos.xlsx")
detalle = pd.read_excel("Detalle_Medicacion.xlsx")

In [ ]:
def limpieza_estructural(df):
    # Elimina filas completamente vacías.
    df = df.dropna(how="all")

    # Elimina columnas completamente vacías.
    df = df.dropna(axis=1, how="all")

    # Elimina filas duplicadas exactas.
    df = df.drop_duplicates() # Por defecto, conserva la primera ocurrenci

    # Retorna una copia independiente del DataFrame.
    return df.copy()


# Aplicamos la misma limpieza a las cuatro tablas.
pacientes = limpieza_estructural(pacientes)
atenciones = limpieza_estructural(atenciones)
medicamentos = limpieza_estructural(medicamentos)
detalle = limpieza_estructural(detalle)

# Convertimos fecha a tipo datetime para poder ordenarla y resumirla.
atenciones["fecha"] = pd.to_datetime(atenciones["fecha"])

# Revisamos las dimensiones resultantes.
print("Pacientes:", pacientes.shape)
print("Atenciones:", atenciones.shape)
print("Medicamentos:", medicamentos.shape)
print("Detalle:", detalle.shape)

## <font color=green>2. Reconstruir la base integrada a nivel paciente</font>

Esta sección es un repaso de integración. La unidad de análisis final será:

> **una fila = un paciente**

In [ ]:
tablas = {
    "Pacientes": pacientes,
    "Atenciones": atenciones,
    "Medicamentos": medicamentos,
    "Detalle_Medicacion": detalle
}
for nombre, df in tablas.items():
    print(f"\n--- {nombre} ---")

    # Revisa tipos de datos
    display(df.dtypes.to_frame("tipo"))
    # Revisa valores faltantes
    display(df.isna().sum().to_frame("faltantes"))
    display(df.head(3))

Todavía **no** tratamos `espera_min`. Por eso, `espera_promedio` se calcula con los datos tal como están y puede permanecer faltante para algún paciente.

In [ ]:
# Convertimos reingreso a una representación binaria.
# Esto nos permitirá resumir luego si el paciente tuvo al menos un reingreso.
atenciones["reingreso_bin"] = atenciones["reingreso"].map({"No": 0, "Sí": 1})

# Resumimos todas las atenciones de cada paciente.
resumen_atenciones = (
    atenciones
    .groupby("id_paciente")
    .agg(
        n_atenciones=("id_atencion", "nunique"),
        fecha_primera_atencion=("fecha", "min"),
        fecha_ultima_atencion=("fecha", "max"),
        espera_promedio=("espera_min", "mean"), # mean() ignora NaN por defecto.

        #¿El paciente tuvo al menos un reingreso en alguna de sus atenciones?
        tuvo_reingreso=("reingreso_bin", "max")# Si alguna atención tuvo reingreso=1, el máximo será 1.
    )
    .reset_index()
)


# Enriquecemos cada prescripción con información del medicamento
# y relacionamos la atención con el paciente correspondiente.
detalle_medicacion = (
    detalle
    .merge(
        medicamentos[["id_medicamento", "familia"]],
        on="id_medicamento",
        how="left",
        validate="many_to_one"
    )
    .merge(
        atenciones[["id_atencion", "id_paciente"]],
        on="id_atencion",
        how="left",
        validate="many_to_one"
    )
)


# Resumimos la información de medicación a nivel paciente.
resumen_medicacion = (
    detalle_medicacion
    .groupby("id_paciente")
    .agg(
        # Total de prescripciones registradas.
        n_prescripciones=("id_medicamento", "size"),

        # Número de medicamentos diferentes recibidos.
        n_medicamentos_distintos=("id_medicamento", "nunique"),
    )
    .reset_index()
)

# Integramos los resúmenes con la tabla Pacientes.
# Al finalizar, cada fila seguirá representando un paciente.
base_paciente = (
    pacientes
    .merge(resumen_atenciones, on="id_paciente", how="left", validate="one_to_one")
    .merge(resumen_medicacion, on="id_paciente", how="left", validate="one_to_one")
)

display(base_paciente)

**Nota**: Se conserva `dosis_mg` en Detalle_Medicacion
Los faltantes de esa variable se revisan en su granularidad original. Pero en este caso eso como no es conceptualmente claro, porque las dosis pueden corresponder a medicamentos distintos y no necesariamente ser comparables entre sí,  decidimos no imputar automáticamente.

De forma similar `presentacion` de la tabla Medicamentos tampoco la incluiría en base_paciente, porque describe al medicamento, no al paciente.

## <font color=green>3. Detectar faltantes que permanecen</font>

Antes de decidir qué hacer, primero debemos saber **dónde están los faltantes y cuántos son**.

In [ ]:
# Contamos los valores faltantes de cada variable.
faltantes = base_paciente.isna().sum()

# Mostramos solo las variables que tienen al menos un faltante.
print(faltantes[faltantes > 0])

In [ ]:
# Identificamos las columnas que tienen valores faltantes.
columnas_faltantes = (
    faltantes[faltantes > 0]
    .index
    .tolist()
)

# Mostramos solo los pacientes que presentan al menos un faltante.
display(
    base_paciente.loc[
        base_paciente[columnas_faltantes]
        .isna()
        .any(axis=1),

        ["id_paciente"] + columnas_faltantes
    ]
)

**¿Encontrar un valor faltante `NaN` significa que siempre debemos reemplazarlo por algún valor? ¿Qué deberíamos analizar antes de decidir?**

> No. Primero debemos entender la variable, la causa posible del faltante y si una imputación tendría sentido conceptual. En algunos casos se puede imputar; en otros se puede eliminar el registro, excluir la variable o incluso conservar el faltante.



## <font color=green>4. Imputación numérica antes de agregar</font>

Volvemos a la tabla `Atenciones`.

Aquí el faltante pertenece a `espera_min`, cuya granularidad es:

> **una fila = una atención**
### <font color=blue>Caso A: imputar `espera_min` antes de agregar</font>

In [ ]:
# Revisamos primero la distribución de espera_min.
display(atenciones["espera_min"].describe())

# El histograma permite observar la forma de la distribución.
plt.subplots(figsize=(4, 3))
sns.histplot(
    data=atenciones,
    x="espera_min",
    bins=7
)
plt.show()

La distribución presenta cierta simetría y algunos valores altos, por lo que utilizaremos la mediana, que es menos sensible a valores extremos.

In [ ]:
# Calculamos la mediana de espera_min.
mediana_espera = atenciones["espera_min"].median()

# Trabajamos sobre una copia para conservar la variable original.
atenciones_imp = atenciones.copy()

# Creamos una nueva variable imputada.
# Si espera_min tiene un valor, se conserva.
# Si está vacío, se reemplaza por la mediana.
atenciones_imp["espera_min_imp"] = (
    atenciones_imp["espera_min"].fillna(mediana_espera)
)

# Comparamos la variable original y la imputada.
display(atenciones_imp[["id_atencion","id_paciente","espera_min","espera_min_imp"]])

In [ ]:
# Después de tratar el faltante a nivel atención,
# Agregamos a nivel paciente y calculamos el promedio por paciente.

resumen_atenciones_imp = (
    atenciones_imp.groupby("id_paciente")
    .agg(espera_promedio_preimp=("espera_min_imp","mean"))
    .reset_index()
)

display(resumen_atenciones_imp)

**¿Qué variable estamos imputando en este punto: el tiempo de espera de una atención (`espera_min`) o el tiempo de espera promedio de un paciente (`espera_promedio`)? ¿Por qué?**

> Estamos imputando `espera_promedio` del paciente, porque la imputación se está realizando después de haber agrupado las atenciones por paciente.

## <font color=green>5. Imputación numérica después de agregar</font>

Ahora trabajamos directamente sobre `base_paciente`.

Aquí el faltante pertenece a `espera_promedio`, cuya granularidad es:

> **una fila = un paciente**

### <font color=blue>Caso B: imputar `espera_promedio` después de agregar</font>


In [ ]:
# Revisamos la variable agregada.
display(base_paciente["espera_promedio"].describe())

# Observamos su distribución.
plt.subplots(figsize=(4, 3))
sns.histplot(
    data=base_paciente,
    x="espera_promedio",
    bins=7
)
plt.show()

La mayor parte de los valores de espera_promedio se concentra aproximadamente entre 20 y 32 minutos. Se observa un valor a los aprox los 42 minutos, lo que genera cierta asimetría hacia la derecha.

Entonces dada la distribución con cierta asimetría y un valor alto, usar la mediana puede ser una medida más robusta para realizar una imputación simple.

In [ ]:
# Calculamos la mediana de espera_promedio.
mediana_promedio = ( base_paciente["espera_promedio"].median())

# Creamos una nueva variable imputada a nivel paciente.
base_paciente["espera_promedio_imp"] = (
    base_paciente["espera_promedio"].fillna(mediana_promedio)
)

# Comparamos la variable original con la nueva.
display(base_paciente[["id_paciente","espera_promedio","espera_promedio_imp"]])

Imputar antes y después de agregar **no es equivalente**:

- antes de agregar se recupera un valor a nivel **atención**;
- después de agregar se recupera una variable a nivel **paciente**.

Para la matriz de esta sesión utilizaremos `espera_promedio_imp`.

## <font color=green>6. ¿Qué pasaría si eliminamos?</font>

No vamos a reemplazar nuestra base principal. Solo evaluaremos cuánto perderíamos al aplicar `dropna()`.

In [ ]:
# CASO A: eliminar a nivel atención.
#         Elimina filas de atenciones sin tiempo de espera

# Número de atenciones antes de eliminar.
n_antes = atenciones.shape[0]

# Eliminamos solo las atenciones donde falta espera_min.
# subset indica en qué variable se buscará el faltante.
atenciones_sin_espera = atenciones.dropna(subset=["espera_min"]).copy()

# Número de atenciones que quedan.
n_despues = atenciones_sin_espera.shape[0]

print("Atenciones antes:", n_antes)
print("Atenciones después:", n_despues)
print("Atenciones eliminadas:", n_antes - n_despues)

In [ ]:
# CASO B: eliminar en la base a nivel paciente.
#         Elimina filas de pacientes con espera_promedio faltante

# Número de pacientes antes de eliminar.
n_antes = base_paciente.shape[0]

# Eliminamos pacientes cuyo espera_promedio está faltante.
base_reducida = base_paciente.dropna(subset=["espera_promedio"]).copy()

# Número de pacientes que quedan.
n_despues = base_reducida.shape[0]

# Calculamos cuántos pacientes se perderían.
n_eliminados = n_antes - n_despues

# Calculamos el porcentaje que representan.
porcentaje = (n_eliminados / n_antes * 100)

print("Pacientes antes:", n_antes)
print("Pacientes después:", n_despues)
print("Pacientes eliminados:", n_eliminados)
print(f"Porcentaje eliminado: {porcentaje:.1f}%")

**Aprendizaje:** eliminar una fila en `Atenciones` elimina una atención; eliminar una fila en `base_paciente` elimina un paciente completo de la unidad de análisis.

## <font color=green> 7. Faltante categórico: `distrito`</font>

Compararemos dos decisiones distintas:

- imputar con la moda;
- conservar explícitamente que el dato es **desconocido**.

In [ ]:
# Revisamos las frecuencias de distrito.
# dropna=False hace visible también la categoría faltante.
base_paciente["distrito"].value_counts(dropna=False)

In [ ]:
# mode() devuelve la moda o las modas de la variable.
modas = base_paciente["distrito"].mode()

display(modas)

moda = modas[0] #selecciona la primera moda.

print("Primera moda:", moda)

In [ ]:
# Opción A: sustituir el faltante por la categoría más frecuente.
base_paciente["distrito_moda"] = base_paciente["distrito"].fillna(moda)

# Opción B: mantener explícitamente que no conocemos el distrito.
base_paciente["distrito_imp"] = base_paciente["distrito"].fillna("Desconocido")

# Comparamos ambas decisiones.
display(base_paciente[["id_paciente","distrito","distrito_moda","distrito_imp"]])

**¿cuál de las dos alternativas asigna al paciente un distrito sin evidencia directa?**

En este caso preferiremos `"Desconocido"`.

## <font color=green> 8. `dosis_mg`: detectar no significa imputar</font>

`dosis_mg` es numérica, pero antes de imputarla debemos comprobar si su significado y unidad son comparables entre medicamentos y presentaciones.

In [ ]:
# Mostramos las prescripciones donde falta dosis_mg.
display(detalle[detalle["dosis_mg"].isna()])

In [ ]:
# Incorporamos información del medicamento para interpretar el faltante.
detalle_contexto = (
    detalle
    .merge(
        medicamentos[["id_medicamento","nombre","presentacion"]],
        on="id_medicamento",
        how="left",
        validate="many_to_one"
    )
)

# Mostramos solo los registros con dosis faltante
# junto con las variables necesarias para entender su contexto.
display(
    detalle_contexto.loc[
        detalle_contexto["dosis_mg"].isna(),
        ["id_atencion","id_medicamento","nombre","presentacion","dosis_mg", "frecuencia_dia","via"]
    ]
)

### Decisión

**No imputaremos `dosis_mg`** porque no podemos garantizar que la variable y su unidad sean comparables entre medicamentos y presentaciones.

Además, `dosis_mg` no es necesaria para la matriz analítica actual.

## <font color=green>9. Excluir una variable que no utilizaremos</font>

In [ ]:
# Si presentacion no es necesaria para el objetivo actual,
# podemos excluirla explícitamente.

medicamentos_reducidos = medicamentos.drop(columns=["presentacion"]).copy()

display(medicamentos_reducidos)

Recuerde:

- `dropna()` elimina filas o columnas según la presencia de faltantes;
- `drop(columns=[...])` excluye explícitamente una variable.

## <font color=green>10. Codificación binaria</font>

In [ ]:
# Codificamos asegurado:
# No -> 0
# Sí -> 1
base_paciente["asegurado_bin"] = base_paciente["asegurado"].map({"No": 0,"Sí": 1 })

# Codificamos sexo:
# F -> 0
# M -> 1
base_paciente["sexo_bin"] = base_paciente["sexo"].map({"F": 0,"M": 1})

# Comparamos las variables originales y codificadas.
display(
    base_paciente[
        [
            "sexo",
            "sexo_bin",
            "asegurado",
            "asegurado_bin"
        ]
    ]
)

Los valores 0 y 1 son **códigos**. No indican que una categoría sea mayor que la otra.

## <font color=green>11. Codificación nominal con one-hot encoding</font>



Ejemplo de one-hot encoding: usaremos `distrito` únicamente para ilustrar cómo se transforma una variable nominal con varias categorías. En un proyecto real, antes de aplicar esta codificación se debe evaluar si la variable es relevante y cuántas categorías tiene.

Porque en la práctica, distrito puede generar muchas columnas y no necesariamente ser una buena característica para el análisis.

In [ ]:
# Creamos columnas indicadoras mediante one-hot encoding.
# La variable nominal distrito se reemplaza por una columna 0/1
# para cada categoría.

base_transformada = pd.get_dummies(
    base_paciente,
    columns=["distrito"],
    prefix="distrito",
    dtype=int
)

# Identificamos automáticamente las nuevas columnas creadas.
columnas_one_hot = [
    columna
    for columna in base_transformada.columns
    if columna.startswith("distrito_")
]

print(columnas_one_hot)

In [ ]:
# Mostramos el resultado de la codificación.
display(
    base_transformada[
        ["id_paciente"] + columnas_one_hot
    ]
)

**¿conviene siempre aplicar one-hot encoding a cualquier variable nominal?**

>No. Depende de su relevancia para el análisis y del número de categorías. Si tiene muchas categorías, el número de columnas puede crecer considerablemente.

## <font color=green>12. Primera matriz analítica</font>

El identificador se conserva para trazabilidad, pero **no se usa como característica analítica**.

In [ ]:
# Conservamos id_paciente por separado.
id_paciente = (
    base_transformada["id_paciente"]
    .copy()
)

# Seleccionamos las variables que sí serán características analíticas.
columnas_analiticas = [
    "edad",
    "sexo_bin",
    "asegurado_bin",
    "espera_promedio_imp",
    "n_atenciones",
    "tuvo_reingreso",
    "n_prescripciones",
    "n_medicamentos_distintos"
]

# Añadimos las columnas generadas por one-hot encoding.
columnas_analiticas = (
    columnas_analiticas
    + columnas_one_hot
)

# Construimos la matriz analítica.
matriz_analitica = (
    base_transformada[
        columnas_analiticas
    ]
    .copy()
)

display(matriz_analitica)

## <font color=green>13. Verificación final</font>

In [ ]:
# Revisamos las dimensiones.
print("Dimensiones:", matriz_analitica.shape)

# Revisamos si quedan faltantes.
print("\nFaltantes por variable:")
print(matriz_analitica.isna().sum())

# Revisamos los tipos de datos.
print("\nTipos de datos:")
print(matriz_analitica.dtypes)

In [ ]:
# Identificamos columnas numéricas de la matriz
columnas_numericas = (
    matriz_analitica #es el DataFrame
    .select_dtypes(include=[np.number]) # selecciona columnas numéricas
    .columns  # obtiene sus nombres
    .tolist() #los convierte en lista
)
print(columnas_numericas)

# Verificamos que el número de filas de la matriz analitica
# coincida con el número de pacientes conservados.
print("Número de pacientes:",id_paciente.shape[0])

print("Número de filas de la matriz:",matriz_analitica.shape[0])